<a href="https://colab.research.google.com/github/broadinstitute/missense-pfes/blob/main/compute_PFES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
if 'google.colab' in sys.modules:
  %pip install g2papi
import g2papi


In [2]:
# @title # Input your gene/protein (HGNC symbol/UniProt accession) and a variant (e.g. M1V)
# @markdown Forms support many types of fields.

gene = 'TGFB1'  # @param {type: "string"}
uniprot = 'P01137'  # @param {type: "string"}
variant = 'C33Y'  # @param {type: "string"}

## Import protein features from Genomics 2 Proteins portal via g2p

In [3]:
# Get protein features as a pandas dataframe
protein_features = g2papi.get_protein_features(gene, uniprot)
protein_features.fillna('-', inplace=True)
protein_features.rename(columns=lambda c: c.replace(" (UniProt)",""), inplace=True)
protein_features.head(5)

/tmp/ipykernel_3886677/62204457.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '-' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  protein_features.fillna('-', inplace=True)


,residueId,AA,Amino acid residues,Amino acid properties,Secondary structure (PDBe/SIFTS),Secondary structure (DSSP 3-state)*,Secondary structure (DSSP 9-state)*,Accessible surface area (Å²)*,Phi angle (degrees)*,Psi angle (degrees)*,...,Intra-chain Non-bonded interaction (PDB),Intra-chain Non-bonded interaction (AlphaFold2),Intra-chain Disulfide bond (PDB),Intra-chain Disulfide bond (AlphaFold2),Intra-chain Salt bridge (PDB),Intra-chain Salt bridge (AlphaFold2),Inter-chain Hydrogen bond (PDB),Inter-chain Non-bonded interaction (PDB),Inter-chain Disulfide bond (PDB),Inter-chain Salt bridge (PDB)
0,1,M,Methionine,Aliphatic,-,C (loop/coil),C (loop/coil),243,360.0,150.7,...,-,-,-,-,-,-,-,-,-,-
1,2,P,Proline,"Special, No backbone hydrogen",-,C (loop/coil),C (loop/coil),103,-90.0,-172.3,...,-,-,-,-,-,-,-,-,-,-
2,3,P,Proline,"Special, No backbone hydrogen",-,C (loop/coil),C (loop/coil),132,-134.9,149.1,...,-,-,-,-,-,-,-,-,-,-
3,4,S,Serine,Polar/Neutral,-,C (loop/coil),C (loop/coil),111,-83.1,142.0,...,-,-,-,-,-,-,-,-,-,-
4,5,G,Glycine,"Special, lack of a chiral carbon, smallest ami...",-,C (loop/coil),C (loop/coil),69,88.2,118.5,...,-,-,-,-,-,-,-,-,-,-


## Retreive PANTHER protein class from G2P metadata

In [4]:
import requests
import pandas as pd
import numpy as np
from io import StringIO

# Construct the public URL for the Google Cloud Storage object
bucket_name = 'g2p-portal'
file_path = 'portal_data/2026_q1_data/uniprot_metadata.tsv'
gcs_public_url = f'https://storage.googleapis.com/{bucket_name}/{file_path}'

try:
    response = requests.get(gcs_public_url)
    response.raise_for_status()  # Raise an exception for HTTP errors (4xx or 5xx)

    file_content = response.text

    # Load the file_content into a pandas DataFrame
    df_uniprot_metadata = pd.read_csv(StringIO(file_content), sep='\t')

    # Extract UniProt ID and PANTHER_protein_class
    uniprot_panther_data = df_uniprot_metadata[['UniprotKB_Entry', 'PANTHER_protein_class']]

    # Filter the uniprot_panther_data DataFrame using the uid (assuming 'uid' is defined)
    panther_class_for_uid = uniprot_panther_data[uniprot_panther_data['UniprotKB_Entry'] == uniprot]['PANTHER_protein_class']

    # Check if a class was found and print it
    if not panther_class_for_uid.empty:
        print(f"PANTHER Protein Class for UniProt ID '{uniprot}':")
        protein_class = panther_class_for_uid.iloc[0]
        display(protein_class)
        
    else:
        print(f"No PANTHER Protein Class found for UniProt ID '{uniprot}'.")

except requests.exceptions.RequestException as e:
    print(f"Error accessing GCS file: {e}")
    print("This might be due to the file/bucket not being publicly accessible or the path being incorrect.")

PANTHER Protein Class for UniProt ID 'P01137':


'intercellular signal molecule'

## Preprocess annotation file and binning 


In [5]:
aa_table = pd.DataFrame([
    {'one': 'A', 'three': 'Ala', 'full': 'Alanine',       'prp': 'Aliphatic'},
    {'one': 'R', 'three': 'Arg', 'full': 'Arginine',      'prp': 'Positively-Charged'},
    {'one': 'N', 'three': 'Asn', 'full': 'Asparagine',    'prp': 'Polar/Neutral'},
    {'one': 'D', 'three': 'Asp', 'full': 'Aspartic acid', 'prp': 'Negatively-Charged'},
    {'one': 'C', 'three': 'Cys', 'full': 'Cysteine',      'prp': 'Special'},
    {'one': 'E', 'three': 'Glu', 'full': 'Glutamic acid', 'prp': 'Negatively-Charged'},
    {'one': 'Q', 'three': 'Gln', 'full': 'Glutamine',     'prp': 'Polar/Neutral'},
    {'one': 'G', 'three': 'Gly', 'full': 'Glycine',       'prp': 'Special'},
    {'one': 'H', 'three': 'His', 'full': 'Histidine',     'prp': 'Positively-Charged'},
    {'one': 'I', 'three': 'Ile', 'full': 'Isoleucine',    'prp': 'Aliphatic'},
    {'one': 'L', 'three': 'Leu', 'full': 'Leucine',       'prp': 'Aliphatic'},
    {'one': 'K', 'three': 'Lys', 'full': 'Lysine',        'prp': 'Positively-Charged'},
    {'one': 'M', 'three': 'Met', 'full': 'Methionine',    'prp': 'Aliphatic'},
    {'one': 'F', 'three': 'Phe', 'full': 'Phenylalanine', 'prp': 'Aromatic'},
    {'one': 'P', 'three': 'Pro', 'full': 'Proline',       'prp': 'Special'},
    {'one': 'S', 'three': 'Ser', 'full': 'Serine',        'prp': 'Polar/Neutral'},
    {'one': 'T', 'three': 'Thr', 'full': 'Threonine',     'prp': 'Polar/Neutral'},
    {'one': 'W', 'three': 'Trp', 'full': 'Tryptophan',    'prp': 'Aromatic'},
    {'one': 'Y', 'three': 'Tyr', 'full': 'Tyrosine',      'prp': 'Aromatic'},
    {'one': 'V', 'three': 'Val', 'full': 'Valine',        'prp': 'Aliphatic'},
])

distance_matrix = {
    'A': {'A': 0, 'R': 112, 'N': 111, 'D': 126, 'C': 195, 'Q': 91, 'E': 107, 'G': 60, 'H': 86, 'I': 94, 'L': 96, 'K': 106, 'M': 84, 'F': 113, 'P': 27, 'S': 99, 'T': 58, 'W': 148, 'Y': 112, 'V': 64},
    'R': {'A': 112, 'R': 0, 'N': 86, 'D': 96, 'C': 180, 'Q': 43, 'E': 54, 'G': 125, 'H': 29, 'I': 97, 'L': 102, 'K': 26, 'M': 91, 'F': 97, 'P': 103, 'S': 110, 'T': 71, 'W': 101, 'Y': 77, 'V': 96},
    'N': {'A': 111, 'R': 86, 'N': 0, 'D': 23, 'C': 139, 'Q': 46, 'E': 42, 'G': 80, 'H': 68, 'I': 149, 'L': 153, 'K': 94, 'M': 142, 'F': 158, 'P': 91, 'S': 46, 'T': 65, 'W': 174, 'Y': 143, 'V': 133},
    'D': {'A': 126, 'R': 96, 'N': 23, 'D': 0, 'C': 154, 'Q': 61, 'E': 45, 'G': 94, 'H': 81, 'I': 168, 'L': 172, 'K': 101, 'M': 160, 'F': 177, 'P': 108, 'S': 65, 'T': 85, 'W': 181, 'Y': 160, 'V': 152},
    'C': {'A': 195, 'R': 180, 'N': 139, 'D': 154, 'C': 0, 'Q': 154, 'E': 170, 'G': 159, 'H': 174, 'I': 198, 'L': 198, 'K': 202, 'M': 196, 'F': 205, 'P': 169, 'S': 112, 'T': 149, 'W': 215, 'Y': 194, 'V': 192},
    'Q': {'A': 91, 'R': 43, 'N': 46, 'D': 61, 'C': 154, 'Q': 0, 'E': 29, 'G': 87, 'H': 24, 'I': 109, 'L': 113, 'K': 53, 'M': 101, 'F': 116, 'P': 76, 'S': 68, 'T': 42, 'W': 130, 'Y': 99, 'V': 96},
    'E': {'A': 107, 'R': 54, 'N': 42, 'D': 45, 'C': 170, 'Q': 29, 'E': 0, 'G': 98, 'H': 40, 'I': 134, 'L': 138, 'K': 56, 'M': 126, 'F': 140, 'P': 93, 'S': 80, 'T': 65, 'W': 152, 'Y': 122, 'V': 121},
    'G': {'A': 60, 'R': 125, 'N': 80, 'D': 94, 'C': 159, 'Q': 87, 'E': 98, 'G': 0, 'H': 98, 'I': 135, 'L': 138, 'K': 127, 'M': 127, 'F': 153, 'P': 42, 'S': 56, 'T': 59, 'W': 184, 'Y': 147, 'V': 109},
    'H': {'A': 86, 'R': 29, 'N': 68, 'D': 81, 'C': 174, 'Q': 24, 'E': 40, 'G': 98, 'H': 0, 'I': 94, 'L': 99, 'K': 32, 'M': 87, 'F': 100, 'P': 77, 'S': 89, 'T': 47, 'W': 115, 'Y': 83, 'V': 84},
    'I': {'A': 94, 'R': 97, 'N': 149, 'D': 168, 'C': 198, 'Q': 109, 'E': 134, 'G': 135, 'H': 94, 'I': 0, 'L': 5, 'K': 102, 'M': 10, 'F': 21, 'P': 95, 'S': 142, 'T': 89, 'W': 61, 'Y': 33, 'V': 29},
    'L': {'A': 96, 'R': 102, 'N': 153, 'D': 172, 'C': 198, 'Q': 113, 'E': 138, 'G': 138, 'H': 99, 'I': 5, 'L': 0, 'K': 107, 'M': 15, 'F': 22, 'P': 98, 'S': 145, 'T': 92, 'W': 61, 'Y': 36, 'V': 32},
    'K': {'A': 106, 'R': 26, 'N': 94, 'D': 101, 'C': 202, 'Q': 53, 'E': 56, 'G': 127, 'H': 32, 'I': 102, 'L': 107, 'K': 0, 'M': 95, 'F': 102, 'P': 103, 'S': 121, 'T': 78, 'W': 110, 'Y': 85, 'V': 97},
    'M': {'A': 84, 'R': 91, 'N': 142, 'D': 160, 'C': 196, 'Q': 101, 'E': 126, 'G': 127, 'H': 87, 'I': 10, 'L': 15, 'K': 95, 'M': 0, 'F': 28, 'P': 87, 'S': 135, 'T': 81, 'W': 67, 'Y': 36, 'V': 21},
    'F': {'A': 113, 'R': 97, 'N': 158, 'D': 177, 'C': 205, 'Q': 116, 'E': 140, 'G': 153, 'H': 100, 'I': 21, 'L': 22, 'K': 102, 'M': 28, 'F': 0, 'P': 114, 'S': 155, 'T': 103, 'W': 40, 'Y': 22, 'V': 50},
    'P': {'A': 27, 'R': 103, 'N': 91, 'D': 108, 'C': 169, 'Q': 76, 'E': 93, 'G': 42, 'H': 77, 'I': 95, 'L': 98, 'K': 103, 'M': 87, 'F': 114, 'P': 0, 'S': 74, 'T': 38, 'W': 147, 'Y': 110, 'V': 68},
    'S': {'A': 99, 'R': 110, 'N': 46, 'D': 65, 'C': 112, 'Q': 68, 'E': 80, 'G': 56, 'H': 89, 'I': 142, 'L': 145, 'K': 121, 'M': 135, 'F': 155, 'P': 74, 'S': 0, 'T': 58, 'W': 177, 'Y': 144, 'V': 124},
    'T': {'A': 58, 'R': 71, 'N': 65, 'D': 85, 'C': 149, 'Q': 42, 'E': 65, 'G': 59, 'H': 47, 'I': 89, 'L': 92, 'K': 78, 'M': 81, 'F': 103, 'P': 38, 'S': 58, 'T': 0, 'W': 128, 'Y': 92, 'V': 69},
    'W': {'A': 148, 'R': 101, 'N': 174, 'D': 181, 'C': 215, 'Q': 130, 'E': 152, 'G': 184, 'H': 115, 'I': 61, 'L': 61, 'K': 110, 'M': 67, 'F': 40, 'P': 147, 'S': 177, 'T': 128, 'W': 0, 'Y': 37, 'V': 88},
    'Y': {'A': 112, 'R': 77, 'N': 143, 'D': 160, 'C': 194, 'Q': 99, 'E': 122, 'G': 147, 'H': 83, 'I': 33, 'L': 36, 'K': 85, 'M': 36, 'F': 22, 'P': 110, 'S': 144, 'T': 92, 'W': 37, 'Y': 0, 'V': 55},
    'V': {'A': 64, 'R': 96, 'N': 133, 'D': 152, 'C': 192, 'Q': 96, 'E': 121, 'G': 109, 'H': 84, 'I': 29, 'L': 32, 'K': 97, 'M': 21, 'F': 50, 'P': 68, 'S': 124, 'T': 69, 'W': 88, 'Y': 55, 'V': 0},
}



In [6]:
import numpy as np
import pandas as pd

# ── Constants ──────────────────────────────────────────────────────────────────

MAX_ASA = {                          # Tien et al. 2013 (DOI:10.1371/journal.pone.0080635)
    'A': 129.0, 'R': 274.0, 'N': 195.0, 'D': 193.0, 'C': 167.0,
    'E': 223.0, 'Q': 225.0, 'G': 104.0, 'H': 224.0, 'I': 197.0,
    'L': 201.0, 'K': 236.0, 'M': 224.0, 'F': 240.0, 'P': 159.0,
    'S': 155.0, 'T': 172.0, 'W': 285.0, 'Y': 263.0, 'V': 174.0,
}

SS9_LABELS = ['B', 'E', 'G', 'H', 'I', 'P', 'C', 'S', 'T']

RSA_BINS   = [0, 0.05, 0.25, 0.50, 0.75, 1.01]
RSA_LABELS = ['Core', 'Buried', 'Medium-buried', 'Medium-exposed', 'Exposed']

PLDDT_BINS   = [0, 50, 70, 90, 100.01]
PLDDT_LABELS = ['Very low', 'Low', 'High', 'Very high']

PCHEM_CLASSES = [
    'Aliphatic', 'Aromatic', 'Polar/Neutral',
    'Positively-Charged', 'Negatively-Charged', 'Special',
]

FUNCTION_FEATURES = ['Active site', 'Binding site', 'Site', 'DNA binding', 'Zinc finger']

DOMAIN_FEATURES = [
    'Region/Disordered', 'Region/Interaction', 'Region/Others',
    'Motif', 'Coiled coil', 'Compositional bias', 'Repeat', 'Domain',
    'Topological domain', 'Transmembrane', 'Intramembrane',
    'Signal', 'Transit peptide', 'Propeptide', 'Peptide', 'Chain',
]

PTM_FEATURES = [
    'Acetylation', 'Methylation', 'Phosphorylation', 'SUMOylation',
    'Ubiquitination', 'O-GalNAc/GlcNAc', 'Lipidation', 'Glycosylation',
    'Cross-link', 'Modified residue',
]

FEATURE_COLS = (
    [f'RefAA:{c}' for c in PCHEM_CLASSES] +
    [f'SS:{s}'    for s in SS9_LABELS] +
    [f'RSA:{b}'   for b in RSA_LABELS] +
    [f'pLDDT:{b}' for b in PLDDT_LABELS] +
    ['PI:HB_intra', 'PI:SB_intra', 'PI:DS_intra', 'PI:NB_intra'] +
    [f'Function:{f}'     for f in FUNCTION_FEATURES] +
    [f'Domain:{d}'       for d in DOMAIN_FEATURES] +
    [f'Modification:{m}' for m in PTM_FEATURES] +
    ['PPI:HB_inter', 'PPI:SB_inter', 'PPI:DS_inter', 'PPI:NB_inter']
)

assert len(FEATURE_COLS) == 63, f"Expected 63, got {len(FEATURE_COLS)}"


# ── Helpers ────────────────────────────────────────────────────────────────────

def _parse_ss9(val):
    if val == '-' or pd.isna(val):
        return None
    s = str(val).strip()
    return s[0] if s[0] in SS9_LABELS else None

def _parse_pchem(val):
    if val == '-' or pd.isna(val):
        return None
    for cls in PCHEM_CLASSES:
        if str(val).startswith(cls):
            return cls
    return None

def _is_annotated(val):
    return str(val).strip() != '-'


# ── Pre-processing helpers ─────────────────────────────────────────────────────

def _combine_glyco(row):
    """Merge O-GalNAc and O-GlcNAc columns into a single value."""
    galNAc = str(row.get('O-GalNAc', '-')).strip()
    glcNAc = str(row.get('O-GlcNAc', '-')).strip()
    if galNAc == '-' and glcNAc == '-':
        return '-'
    if galNAc == '-':
        return glcNAc
    if glcNAc == '-':
        return galNAc
    return f'{galNAc},{glcNAc}'


def _route_disulfide_uniprot(val, ds_intra, ds_inter):
    """
    Route a UniProt 'Disulfide bond' annotation to DS_intra or DS_inter
    based on whether 'Interchain' appears after the colon.
    val format: 'residue:note'  e.g. '77:Interchain' or '77:...'
    """
    if str(val).strip() == '-':
        return ds_intra, ds_inter
    parts = str(val).split(':', 1)
    if len(parts) == 2 and 'interchain' in parts[1].lower():
        ds_inter = val if ds_inter == '-' else f'{ds_inter},{val}'
    else:
        ds_intra = val if ds_intra == '-' else f'{ds_intra},{val}'
    return ds_intra, ds_inter

def _classify_region(val):
    """
    Parse a Region annotation (semicolon-separated entries) into sub-types.
    Returns a set of: 'Region/Disordered', 'Region/Interaction', 'Region/Others'.

    Examples
    --------
    'Disordered'                  -> {'Region/Disordered'}
    'Disordered;Interact with X'  -> {'Region/Disordered', 'Region/Interaction'}
    'ZP-C'                        -> {'Region/Others'}
    '-'                           -> set()
    """
    matched = set()
    if str(val).strip() == '-':
        return matched
    for entry in str(val).split(';'):
        entry = entry.strip()
        if not entry:
            continue
        if entry == 'Disordered':
            matched.add('Region/Disordered')
        elif 'interact' in entry.lower():
            matched.add('Region/Interaction')
        else:
            matched.add('Region/Others')
    return matched


# ── Main function ──────────────────────────────────────────────────────────────

def annotate_protein_features(pf): #, aa_table):
    """
    Convert g2papi protein_features dataframe into a boolean feature matrix.

    Parameters
    ----------
    pf       : pd.DataFrame   output of g2papi.get_protein_features()
    aa_table : pd.DataFrame   with columns ['one', 'prp']

    Returns
    -------
    df : pd.DataFrame
        Same row index as pf, columns = ['ResID', 'RefAA'] + 63 feature codes,
        dtype bool (except ResID and RefAA).
        AAchange and Grantham columns are added later per alt AA.
    """
    df = pd.DataFrame(False, index=pf.index, columns=['ResID', 'RefAA'] + FEATURE_COLS)
    df['ResID'] = pf['residueId']
    df['RefAA']  = pf['AA']

    # ── Secondary structure (9) ───────────────────────────────────────────────
    ss_col = 'Secondary structure (DSSP 9-state)*'
    if ss_col in pf.columns:
        parsed = pf[ss_col].apply(_parse_ss9)
        for s in SS9_LABELS:
            df[f'SS:{s}'] = parsed == s

    # ── RSA (5) ───────────────────────────────────────────────────────────────
    asa_col = 'Accessible surface area (Å²)*'
    if asa_col in pf.columns:
        asa  = pd.to_numeric(pf[asa_col], errors='coerce')
        rsa  = (asa / pf['AA'].map(MAX_ASA)).clip(0, 1)
        bins = pd.cut(rsa, bins=RSA_BINS, labels=RSA_LABELS, right=False)
        for b in RSA_LABELS:
            df[f'RSA:{b}'] = bins == b

    # ── pLDDT (4) ─────────────────────────────────────────────────────────────
    plddt_col = 'AlphaFold confidence (pLDDT)'
    if plddt_col in pf.columns:
        plddt = pd.to_numeric(pf[plddt_col], errors='coerce')
        bins  = pd.cut(plddt, bins=PLDDT_BINS, labels=PLDDT_LABELS, right=False)
        for b in PLDDT_LABELS:
            df[f'pLDDT:{b}'] = bins == b

    # ── Ref AA physicochemical class (6) ──────────────────────────────────────
    aa2prp  = aa_table.set_index('one')['prp'].to_dict()
    ref_cls = pf['AA'].map(aa2prp)
    for c in PCHEM_CLASSES:
        df[f'RefAA:{c}'] = ref_cls == c

    # ── Intra-chain interactions (4, OR logic PDB + AF2) ──────────────────────
    intra_map = {
        'PI:HB_intra': ['Intra-chain Hydrogen bond (PDB)',         'Intra-chain Hydrogen bond (AlphaFold2)'],
        'PI:SB_intra': ['Intra-chain Salt bridge (PDB)',           'Intra-chain Salt bridge (AlphaFold2)'],
        'PI:DS_intra': ['Intra-chain Disulfide bond (PDB)',        'Intra-chain Disulfide bond (AlphaFold2)'],
        'PI:NB_intra': ['Intra-chain Non-bonded interaction (PDB)','Intra-chain Non-bonded interaction (AlphaFold2)'],
    }
    for feat, cols in intra_map.items():
        avail = [c for c in cols if c in pf.columns]
        if avail:
            df[feat] = pf[avail].apply(
                lambda row: any(_is_annotated(v) for v in row), axis=1
            )

    # ── UniProt 'Disulfide bond' → route to DS_intra or DS_inter ──────────────
    if 'Disulfide bond' in pf.columns:
        for i, val in enumerate(pf['Disulfide bond']):
            cur_intra = '-' if not df.at[i, 'PI:DS_intra'] else 'annotated'
            cur_inter = '-' if not df.at[i, 'PPI:DS_inter'] else 'annotated'
            new_intra, new_inter = _route_disulfide_uniprot(val, cur_intra, cur_inter)
            df.at[i, 'PI:DS_intra']  = new_intra != '-'
            df.at[i, 'PPI:DS_inter'] = new_inter != '-'

    # ── PPI: inter-chain interactions (4, PDB only) ───────────────────────────
    inter_map = {
        'PPI:HB_inter': 'Inter-chain Hydrogen bond (PDB)',
        'PPI:SB_inter': 'Inter-chain Salt bridge (PDB)',
        'PPI:DS_inter': 'Inter-chain Disulfide bond (PDB)',
        'PPI:NB_inter': 'Inter-chain Non-bonded interaction (PDB)',
    }
    for feat, col in inter_map.items():
        if col in pf.columns:
            df[feat] = df[feat] | pf[col].apply(_is_annotated)

    # ── Function features (5) — each is its own column ───────────────────────
    for f in FUNCTION_FEATURES:
        df[f'Function:{f}'] = pf[f].apply(_is_annotated)

    # ── Domain features (16) — each is its own column ────────────────────────
    # Region is further split into 3 sub-types by keyword
    region_cats = pf['Region'].apply(_classify_region)
    for sub in ['Region/Disordered', 'Region/Interaction', 'Region/Others']:
        df[f'Domain:{sub}'] = region_cats.apply(lambda s: sub in s)

    for d in [d for d in DOMAIN_FEATURES if not d.startswith('Region/')]:
        df[f'Domain:{d}'] = pf[d].apply(_is_annotated)

    # ── PTM features (10) — each is its own column ───────────────────────────
    # O-GalNAc and O-GlcNAc are merged into one feature
    glyco = pf[['O-GalNAc', 'O-GlcNAc']].apply(_combine_glyco, axis=1)
    df['Modification:O-GalNAc/GlcNAc'] = glyco.apply(_is_annotated)

    for m in [m for m in PTM_FEATURES if m != 'O-GalNAc/GlcNAc']:
        df[f'Modification:{m}'] = pf[m].apply(_is_annotated)

    return df

In [7]:
encoded_feature_refaa = annotate_protein_features(protein_features)

# Retrieve precomputed protein class-specific odds ratio table 

In [8]:
import requests
import pandas as pd
from io import StringIO

# Raw URL for the CSV file on GitHub
github_csv_url = 'https://raw.githubusercontent.com/broadinstitute/missense-pfes/refs/heads/main/results/enrichment_OR_by_protein_class.csv'
try:
    response = requests.get(github_csv_url)
    response.raise_for_status() # Raise an exception for HTTP errors (4xx or 5xx)

    # Read the content into a pandas DataFrame
    enrichment_df = pd.read_csv(StringIO(response.text), header=[0, 1], index_col=0)

    print("Successfully loaded the enrichment data. Here's a preview:")
    display(enrichment_df.head())

    # --- Extract odd ratio for a given protein class ---
    # Filter the DataFrame for the desired protein class
    odd_ratio_data = enrichment_df[protein_class][['OR','q_value']]


    # --- Handling infinite and zero values in odds ratios to stabilize log transformation --- 
    max_val = odd_ratio_data.loc[np.isfinite(odd_ratio_data['OR']), 'OR'].max()
    min_val = odd_ratio_data.loc[odd_ratio_data['OR'] > 0, 'OR'].min()
    odd_ratio_data['OR'] = np.where(odd_ratio_data['OR'] == np.inf, max_val, odd_ratio_data['OR'])
    odd_ratio_data['OR'] = np.where(odd_ratio_data['OR'] == 0, min_val, odd_ratio_data['OR'])

    if not odd_ratio_data.empty:
        print(f"\nOdd Ratio for '{protein_class}':")
        # Assuming 'Odd Ratio' is the column name for the odd ratio
        display(odd_ratio_data)
    else:
        print(f"\nNo data found for protein class: '{protein_class}'.")
        print("Please check the exact spelling of the protein class.")

except requests.exceptions.RequestException as e:
    print(f"Error accessing the GitHub CSV file: {e}")
    print("Please ensure the URL is correct and the file is publicly accessible.")

Successfully loaded the enrichment data. Here's a preview:


All                                                             \
               OR     CI_lo     CI_up       p_value       q_value n_case_yes   
feature                                                                        
SS:B     2.095935  1.863285  2.357634  1.042035e-35  1.511684e-35      688.0   
SS:E     1.949960  1.896735  2.004679  0.000000e+00  0.000000e+00    13002.0   
SS:G     1.353493  1.278055  1.433383  6.546386e-25  8.756854e-25     2331.0   
SS:H     1.673580  1.640216  1.707623  0.000000e+00  0.000000e+00    27537.0   
SS:I     3.117067  2.723904  3.566979  7.270310e-67  1.291107e-66      674.0   

                   DNA metabolism protein                      ...  \
        n_ctrl_yes                     OR     CI_lo     CI_up  ...   
feature                                                        ...   
SS:B         470.0               1.198662  0.542820  2.646901  ...   
SS:E       10362.0               2.320431  1.925335  2.796603  ...   
SS:G        2474.0               0.924600  0.634905  1.346477  ...   
SS:H       27431.0               1.788521  1.563988  2.045289  ...   
SS:I         310.0               1.055290  0.450050  2.474476  ...   

           transporter                       unclassified                      \
               q_value n_case_yes n_ctrl_yes           OR     CI_lo     CI_up   
feature                                                                         
SS:B      9.484252e-02       81.0       40.0     2.152166  1.613920  2.869919   
SS:E      9.743241e-03      830.0      499.0     2.737234  2.566682  2.919119   
SS:G      2.442619e-08      468.0      207.0     1.381684  1.198402  1.592996   
SS:H     1.483285e-182     6984.0     3162.0     1.746341  1.665998  1.830557   
SS:I      2.805079e-15      236.0       56.0     4.263885  3.041343  5.977856   

                                                             
               p_value        q_value n_case_yes n_ctrl_yes  
feature                                                      
SS:B      4.300968e-07   6.152774e-07       81.0      110.0  
SS:E     2.270489e-197  1.299224e-196     1971.0     2372.0  
SS:G      1.354697e-05   1.766250e-05      286.0      607.0  
SS:H     1.804233e-116  6.408139e-116     3866.0     7614.0  
SS:I      3.338566e-17   5.928832e-17       83.0       57.0  

[5 rows x 147 columns]


Odd Ratio for 'intercellular signal molecule':


,OR,q_value
feature,,
SS:B,1.104247,9.541433e-01
SS:E,2.591396,1.767627e-26
SS:G,0.641132,8.326383e-02
SS:H,0.668305,1.661662e-04
SS:I,1.103448,1.000000e+00
...,...,...
Modification:Modified residue,0.390294,7.627388e-01
PPI:HB_inter,12.311354,8.045416e-42
PPI:SB_inter,29.859485,1.451966e-07


# Mutational landscape heatmap 

In [16]:
import re
import numpy as np
import pandas as pd
from itertools import product

# ── Grantham distance (already defined earlier, included for completeness) ─────
# distance_matrix = { ... }  # assumed available

# ── AA ordering for heatmap columns ──────────────────────────────────────────
AA_ORDER = list('ACDEFGHIKLMNPQRSTVWY')   # 20 canonical AA, alphabetical

# ── Attribute → feature prefix mapping ───────────────────────────────────────
ATTR_PREFIXES = {
    'Physicochemical': ['RefAA:', 'AAchange:', 'Grantham:'],
    'Structure':       ['SS:', 'RSA:', 'pLDDT:', 'PI:'],
    'Domain':          ['Domain:'],
    'Function':        ['Function:'],
    'Modification':    ['Modification:'],
    'PPI':             ['PPI:'],
}

Q_THRESHOLD = 0.01   # FDR threshold for significant features


# ── Step 1: Build log_OR lookup from odd_ratio_data ───────────────────────────

def get_or_lookup(odd_ratio_data):
    """
    Build log_OR lookup from the already-loaded odd_ratio_data DataFrame.
    inf/zero OR values are already handled upstream.

    Parameters
    ----------
    odd_ratio_data : pd.DataFrame
        Index = feature codes, columns include 'OR' and 'q_value'.
        (output of enrichment_df[protein_class][['OR','q_value']])

    Returns
    -------
    log_ors  : dict  {feature_code -> log(OR)}  for significant features only
    sig_stats: dict  {feature_code -> {'OR', 'q_value'}}
    """
    sig = odd_ratio_data[odd_ratio_data['q_value'] < Q_THRESHOLD].dropna(subset=['OR'])
    log_ors   = {feat: np.log(row['OR']) for feat, row in sig.iterrows()}
    sig_stats = sig[['OR', 'q_value']].to_dict(orient='index')
    return log_ors, sig_stats


# ── Step 3: Add variant-level features to encoded_feature_refaa ──────────────

def add_variant_features(encoded_refaa, alt_aa, aa_table, distance_matrix):
    """
    For a given alt_aa (single letter), add AAchange and Grantham columns
    to a copy of encoded_refaa. Columns are created fresh regardless of
    whether they exist in encoded_refaa.

    Parameters
    ----------
    encoded_refaa   : pd.DataFrame   output of annotate_protein_features()
    alt_aa          : str            single-letter alt AA (e.g. 'Y')
    aa_table        : pd.DataFrame   with columns ['one', 'prp']
    distance_matrix : dict
    """
    df = encoded_refaa.copy()
    one2prp = aa_table.set_index('one')['prp'].to_dict()
    alt_cls = one2prp.get(alt_aa)
    ref_cls = df['RefAA'].map(one2prp)

    # AAchange: one column per (ref_class, alt_class) pair
    for ref_c in PCHEM_CLASSES:
        for alt_c in PCHEM_CLASSES:
            df[f'AAchange:{ref_c}>{alt_c}'] = (ref_cls == ref_c) & (alt_cls == alt_c)

    # Grantham distance bin
    dist_bins   = [0, 50, 100, 150, np.inf]
    dist_labels = ['Mild', 'Moderate', 'Substantial', 'Severe']
    dists = df['RefAA'].apply(lambda r: distance_matrix.get(r, {}).get(alt_aa, np.nan))
    grantham_bin = pd.cut(dists, bins=dist_bins, labels=dist_labels, right=False)
    for lbl in dist_labels:
        df[f'Grantham:{lbl}'] = grantham_bin == lbl

    return df


# ── Step 4: Compute PFES for one encoded row ──────────────────────────────────

def compute_pfes_row(feat_row, log_ors):
    """
    Given a boolean Series (one residue × one alt_aa) and log_ors dict,
    return total PFES and 6 attribute sub-scores.
    """
    scores = {attr: 0.0 for attr in ATTR_PREFIXES}

    for feat, log_or in log_ors.items():
        if feat not in feat_row.index:
            continue
        if not feat_row[feat]:
            continue
        for attr, prefixes in ATTR_PREFIXES.items():
            if any(feat.startswith(p) for p in prefixes):
                scores[attr] += log_or
                break

    scores['PFES'] = sum(scores.values())
    return scores


# ── Step 5: Build full mutational landscape ───────────────────────────────────

def build_landscape(encoded_refaa, odd_ratio_data, aa_table, distance_matrix):
    """
    Compute PFES for every (residue × alt_AA) combination.

    Returns
    -------
    landscape : dict of 2D arrays, shape (n_residues, 20)
        Keys: 'PFES', 'Physicochemical', 'Structure', 'Domain',
              'Function', 'Modification', 'PPI'
    sig_stats : dict  {feature -> stats}  for hover text
    positions : list of residue IDs
    """
    log_ors, sig_stats = get_or_lookup(odd_ratio_data)
    positions = encoded_refaa['ResID'].tolist()
    n = len(positions)
    keys = ['PFES'] + list(ATTR_PREFIXES.keys())
    landscape = {k: np.full((n, len(AA_ORDER)), np.nan) for k in keys}

    for j, alt_aa in enumerate(AA_ORDER):
        enc = add_variant_features(encoded_refaa, alt_aa, aa_table, distance_matrix)
        feat_cols = [c for c in enc.columns if c not in ('ResID', 'RefAA')]
        for i in range(n):
            ref_aa = encoded_refaa['RefAA'].iloc[i]
            if ref_aa == alt_aa:           # synonymous — leave as NaN
                continue
            row = enc.iloc[i][feat_cols]
            s   = compute_pfes_row(row, log_ors)
            for k in keys:
                landscape[k][i, j] = s[k]

    return landscape, sig_stats, positions


# ── Step 6: Parse input variant ───────────────────────────────────────────────

def parse_variant(variant_str):
    """'C77Y' -> (ref='C', pos=77, alt='Y')"""
    m = re.fullmatch(r'([A-Z])(\d+)([A-Z])', variant_str.strip())
    if not m:
        raise ValueError(f"Cannot parse variant: {variant_str}")
    return m.group(1), int(m.group(2)), m.group(3)

## Interactive heatmap of mutational lancscape

In [17]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Attribute display config ───────────────────────────────────────────────────
ATTR_ORDER  = ['Physicochemical', 'Structure', 'Domain', 'Function', 'Modification', 'PPI']
ATTR_2D     = {'Physicochemical'}
ATTR_COLORS = {
    'PFES':            'RdBu_r',
    'Physicochemical': 'RdBu_r',
    'Structure':       'RdBu_r',
    'Domain':          'RdBu_r',
    'Function':        'RdBu_r',
    'Modification':    'RdBu_r',
    'PPI':             'RdBu_r',
}


def _colorbar_from_domain(fig, row_idx):
    axis_key = 'yaxis' if row_idx == 1 else f'yaxis{row_idx}'
    domain   = fig.layout[axis_key].domain
    y_bottom, y_top = domain
    center = (y_bottom + y_top) / 2
    length = (y_top - y_bottom) * 1.1
    return center, length


def _sig_features_for_hover(feat_row, log_ors, sig_stats, prefixes):
    lines = []
    for feat, log_or in log_ors.items():
        if not any(feat.startswith(p) for p in prefixes):
            continue
        if feat not in feat_row.index or not feat_row[feat]:
            continue
        s      = sig_stats.get(feat, {})
        or_val = s.get('OR',      np.nan)
        q_val  = s.get('q_value', np.nan)
        p_val  = s.get('p_value', np.nan)
        lines.append(f"{feat}  OR={or_val:.2f} q={q_val:.2e}")
    return lines if lines else ['(no significant features)']


def build_hover_pfes(landscape, encoded_refaa, log_ors, sig_stats, positions):
    n       = len(positions)
    hover   = np.empty((n, len(AA_ORDER)), dtype=object)
    ref_aas = encoded_refaa['RefAA'].tolist()
    for j, alt_aa in enumerate(AA_ORDER):
        enc       = add_variant_features(encoded_refaa, alt_aa, aa_table, distance_matrix)
        feat_cols = [c for c in enc.columns if c not in ('ResID', 'RefAA')]
        for i in range(n):
            ref, pos = ref_aas[i], positions[i]
            if ref == alt_aa:
                hover[i, j] = f'{ref}{pos}{alt_aa} (synonymous)'
                continue
            parts = [f'<b>{ref}{pos}{alt_aa}</b>', f'PFES = {landscape["PFES"][i, j]:.3f}']
            for attr in ATTR_ORDER:
                parts.append(f'{attr} = {landscape[attr][i, j]:.3f}')
            hover[i, j] = '<br>'.join(parts)
    return hover


def build_hover_attr(attr, landscape, encoded_refaa, log_ors, sig_stats, positions):
    prefixes = ATTR_PREFIXES[attr]
    n        = len(positions)
    ref_aas  = encoded_refaa['RefAA'].tolist()
    hover    = np.empty((n, len(AA_ORDER)), dtype=object)
    if attr in ATTR_2D:
        for j, alt_aa in enumerate(AA_ORDER):
            enc       = add_variant_features(encoded_refaa, alt_aa, aa_table, distance_matrix)
            feat_cols = [c for c in enc.columns if c not in ('ResID', 'RefAA')]
            for i in range(n):
                ref, pos = ref_aas[i], positions[i]
                if ref == alt_aa:
                    hover[i, j] = f'{ref}{pos}{alt_aa} (synonymous)'
                    continue
                feats = _sig_features_for_hover(enc.iloc[i][feat_cols], log_ors, sig_stats, prefixes)
                score = landscape[attr][i, j]
                hover[i, j] = f'<b>{ref}{pos}{alt_aa}</b>  score={score:.3f}<br>' + '<br>'.join(feats)
    else:
        feat_cols = [c for c in encoded_refaa.columns if c not in ('ResID', 'RefAA')]
        for i in range(n):
            ref, pos  = ref_aas[i], positions[i]
            feats     = _sig_features_for_hover(encoded_refaa.iloc[i][feat_cols], log_ors, sig_stats, prefixes)
            score     = np.nanmean(landscape[attr][i, :]) #landscape[attr][i, 0]
            cell_text = f'<b>{ref}{pos}</b>  score={score:.3f}<br>' + '<br>'.join(feats)
            for j in range(len(AA_ORDER)):
                hover[i, j] = cell_text
    return hover

def _add_panel(fig, row_idx, panel, landscape, encoded_refaa, log_ors,
               sig_stats, positions, x_ticks, xi, yi):
    is_2d  = (panel == 'PFES') or (panel in ATTR_2D)
    cscale = ATTR_COLORS.get(panel, 'RdBu_r')
    zdata  = landscape[panel]
    cb_y, cb_len = _colorbar_from_domain(fig, row_idx)
    cb = dict(
        yref='paper', yanchor='middle', y=cb_y,
        xanchor='left', x=1.02,
        len=cb_len, thickness=12,
        title=dict(text=panel, side='right', font=dict(size=9)),
        tickfont=dict(size=8),
    )
    if is_2d:
        hover = (build_hover_pfes(landscape, encoded_refaa, log_ors, sig_stats, positions)
                 if panel == 'PFES' else
                 build_hover_attr(panel, landscape, encoded_refaa, log_ors, sig_stats, positions))
        trace = go.Heatmap(
            z=zdata.T, x=x_ticks, y=AA_ORDER,
            colorscale=cscale, zmid=0,
            text=hover.T, hovertemplate='%{text}<extra></extra>',
            colorbar=cb, showscale=True, name=panel,
        )
    else:
        hover = build_hover_attr(panel, landscape, encoded_refaa, log_ors, sig_stats, positions)
        trace = go.Heatmap(
            z=np.nanmean(zdata, axis=1, keepdims=True).T,
            x=x_ticks, y=[panel], colorscale=cscale, zmid=0,
            text=hover[:, 0:1].T, hovertemplate='%{text}<extra></extra>',
            colorbar=cb, showscale=True, name=panel,
        )
    fig.add_trace(trace, row=row_idx, col=1)
    if xi is not None:
        yi_val = yi if is_2d else 0
        fig.add_shape(
            type='rect',
            x0=xi - 0.5, x1=xi + 0.5,
            y0=yi_val - 0.5, y1=yi_val + 0.5,
            line=dict(color='black', width=2),
            fillcolor='rgba(0,0,0,0)',
            row=row_idx, col=1,
        )

def plot_pfes_overall(fig, landscape, encoded_refaa, log_ors, sig_stats,
                      positions, x_ticks, xi, yi):
    _add_panel(fig, 1, 'PFES', landscape, encoded_refaa, log_ors,
               sig_stats, positions, x_ticks, xi, yi)


def plot_pfes_attributes(fig, landscape, encoded_refaa, log_ors, sig_stats,
                         positions, x_ticks, xi, yi):
    for i, attr in enumerate(ATTR_ORDER, start=2):
        _add_panel(fig, i, attr, landscape, encoded_refaa, log_ors,
                   sig_stats, positions, x_ticks, xi, yi)
        

def plot_pfes_landscape(landscape, sig_stats, encoded_refaa, log_ors,
                        positions, variant, gene=''):
    ref_aa, var_pos, alt_aa = parse_variant(variant) if variant else (None, None, None)
    n = len(positions)
    row_heights      = [4, 4, 1, 1, 1, 1, 1.2]
    vertical_spacing = 0.05
    fig = make_subplots(
        rows=7, cols=1,
        shared_xaxes=True,
        row_heights=row_heights,
        vertical_spacing=vertical_spacing,
        subplot_titles=['PFES'] + ATTR_ORDER,
    )
    x_ticks  = list(range(n))
    x_labels = [str(p) for p in positions]
    try:
        xi = positions.index(var_pos)
    except (ValueError, TypeError):
        xi = None
    try:
        yi = AA_ORDER.index(alt_aa)
    except (ValueError, TypeError):
        yi = None
    plot_pfes_overall(fig, landscape, encoded_refaa, log_ors, sig_stats,
                      positions, x_ticks, xi, yi)
    plot_pfes_attributes(fig, landscape, encoded_refaa, log_ors, sig_stats,
                         positions, x_ticks, xi, yi)
    step = max(1, n // 50)
    fig.update_xaxes(
        tickvals=x_ticks[::step], ticktext=x_labels[::step],
        tickfont=dict(size=7), row=7, col=1,
    )
    fig.update_layout(
        title=dict(text=f'{gene} — PFES mutational landscape  ({variant})', x=0.5),
        height=1000, width=1300,
        margin=dict(l=60, r=160, t=60, b=50),
        showlegend=False,
    )
    return fig

def run_pfes_landscape(encoded_refaa, odd_ratio_data, gene='', variant=None):
    log_ors, sig_stats = get_or_lookup(odd_ratio_data)
    landscape, sig_stats, positions = build_landscape(
        encoded_refaa, odd_ratio_data, aa_table, distance_matrix
    )
    fig = plot_pfes_landscape(
        landscape, sig_stats, encoded_refaa, log_ors,
        positions, variant, gene
    )
    return fig, landscape, positions


In [18]:
fig, landscape, positions = run_pfes_landscape(
    encoded_refaa  = encoded_feature_refaa,
    odd_ratio_data = odd_ratio_data,
    gene           = gene,
    variant        = variant,
)
fig.show()

# Save output

In [13]:
import os

# ── Print PFES for the queried variant ────────────────────────────────────────
ref_aa, var_pos, alt_aa = parse_variant(variant)
xi = positions.index(var_pos)
yi = AA_ORDER.index(alt_aa)

print(f"\n{'='*50}")
print(f"Variant:  {gene}:{variant}")
print(f"PFES:     {landscape['PFES'][xi, yi]:.3f}")
print(f"{'─'*50}")
for attr in ATTR_ORDER:
    print(f"  {attr:<20} {landscape[attr][xi, yi]:.3f}")
print(f"{'─'*50}")
print(f"Protein class: {protein_class}")

# ── Significant features driving the score ────────────────────────────────────
log_ors, sig_stats = get_or_lookup(odd_ratio_data)
enc      = add_variant_features(encoded_feature_refaa, alt_aa, aa_table, distance_matrix)
feat_cols = [c for c in enc.columns if c not in ('ResID', 'RefAA')]
feat_row  = enc.iloc[xi][feat_cols]

sig_hits = [
    {'feature': feat, 'log_OR': log_or,
     'OR': sig_stats[feat]['OR'], 'q_value': sig_stats[feat]['q_value']}
    for feat, log_or in log_ors.items()
    if feat in feat_row.index and feat_row[feat]
]
sig_df = pd.DataFrame(sig_hits).sort_values('log_OR', ascending=False)
print(f"\nSignificant features ({len(sig_df)}):")
display(sig_df.reset_index(drop=True))

# ── Save outputs ──────────────────────────────────────────────────────────────
out_dir = f'pfes_output_{gene}'
os.makedirs(out_dir, exist_ok=True)

# Interactive HTML figure
fig.write_html(os.path.join(out_dir, f'{gene}_{variant}_landscape.html'))

# Variant summary table
sig_df.to_csv(os.path.join(out_dir, f'{gene}_{variant}_significant_features.csv'), index=False)

# Full landscape scores for all positions x alt AAs
rows = []
for i, pos in enumerate(positions):
    for j, aa in enumerate(AA_ORDER):
        if np.isnan(landscape['PFES'][i, j]):
            continue
        row = {'position': pos,
               'ref_aa': encoded_feature_refaa['RefAA'].iloc[i],
               'alt_aa': aa,
               'PFES': landscape['PFES'][i, j]}
        for attr in ATTR_ORDER:
            row[attr] = landscape[attr][i, j]
        rows.append(row)
pd.DataFrame(rows).to_csv(
    os.path.join(out_dir, f'{gene}_full_landscape.csv'), index=False)

print(f"\nOutputs saved to '{out_dir}/'")
print(f"  {gene}_{variant}_landscape.html")
print(f"  {gene}_{variant}_significant_features.csv")
print(f"  {gene}_full_landscape.csv")


Variant:  TGFB1:C33Y
PFES:     17.132
──────────────────────────────────────────────────
  Physicochemical      5.893
  Structure            1.336
  Domain               1.175
  Function             0.000
  Modification         0.000
  PPI                  8.728
──────────────────────────────────────────────────
Protein class: intercellular signal molecule

Significant features (12):


,feature,log_OR,OR,q_value
0,PI:DS_intra,3.962750,52.601795,9.301434e-59
1,PPI:DS_inter,3.962750,52.601795,4.179746e-03
2,AAchange:Special>Aromatic,2.872625,17.683368,2.783995e-28
3,PPI:HB_inter,2.510522,12.311354,8.045416e-42
4,PPI:NB_inter,2.255004,9.535328,1.082395e-40
5,Grantham:Severe,1.909992,6.753032,1.537189e-55
6,RefAA:Special,1.109900,3.034055,1.304964e-34
7,Domain:Region/Others,0.789728,2.202797,8.994370e-03
8,Domain:Chain,0.385563,1.470442,1.517564e-04
9,SS:C,-0.505639,0.603120,1.513477e-08



Outputs saved to 'pfes_output_TGFB1/'
  TGFB1_C33Y_landscape.html
  TGFB1_C33Y_significant_features.csv
  TGFB1_full_landscape.csv
